<a href="https://colab.research.google.com/github/Tshoki-m/Capstone-Research/blob/main/CIFAR100_PEPITA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CIFAR-100 PEPITA reproduction run

This notebook runs one CIFAR-100 experiment using the project’s fully connected PEPITA implementation. It follows the paper’s one-hidden-layer CIFAR-100 settings: 1,024 hidden units, learning rate 0.01, weight decay 1e-5, batch size 64, 100 epochs, and learning-rate drops at epochs 60 and 90. The fixed seed is recorded in the saved configuration.

**Before starting:** in Colab choose **Runtime → Change runtime type → GPU**. When prompted, upload the companion `PEPITA_source_for_Colab.zip` file. It contains the project source needed by the run and excludes datasets, old experiment outputs, and the local virtual environment.

The notebook creates an isolated Python 3.10 environment with the project’s original PyTorch and Lightning versions. This avoids changing the training framework to match Colab’s newer default Python. It also makes a small data-loader compatibility fix for `val_split=0`: the training set remains intact and the project’s existing validation hook continues to read its test loader. The run logs test metrics during training; treat those as monitoring, and report the final test result without using it to tune settings.

In [1]:
from pathlib import Path
import os, sys, platform, subprocess, textwrap, json, time, shutil, zipfile

print('Colab Python:', sys.version.split()[0])
print('Runtime:', platform.platform())
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'], text=True, capture_output=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError('No GPU detected. In Colab, select Runtime > Change runtime type > GPU, reconnect, then rerun this cell.')
print('GPU:', gpu.stdout.strip())
!nvidia-smi

Colab Python: 3.13.16
Runtime: Linux-6.6.159+-x86_64-with-glibc2.39
GPU: Tesla T4, 7.5
Sat Oct 31 15:02:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |          

## Upload the project source

Upload the companion `PEPITA_source_for_Colab.zip` archive. The notebook will locate the folder containing `scripts/train.py`.


In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one ZIP file containing the PEPITA project source.')
zip_name = next(iter(uploaded))
if not zip_name.lower().endswith('.zip'):
    raise ValueError('Please upload a .zip archive of the PEPITA project source.')

extract_root = Path('/content/pepita_upload')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(extract_root)
train_scripts = list(extract_root.rglob('scripts/train.py'))
if len(train_scripts) != 1:
    raise RuntimeError(f'Expected one scripts/train.py in the ZIP; found {len(train_scripts)}. Rebuild the ZIP with one PEPITA project folder.')
PROJECT = train_scripts[0].parent.parent.resolve()
if not (PROJECT / 'pepita' / 'core' / 'trainer.py').exists():
    raise RuntimeError('The ZIP is missing pepita/core/trainer.py. Include the complete pepita/ and scripts/ source folders.')
os.chdir(PROJECT)
print('Project folder:', PROJECT)

## Create the compatible training environment

Colab’s default Python can be newer than this project supports. The isolated Python 3.10 environment preserves the original package versions while using the selected Colab GPU.

In [ ]:
import os, subprocess, pathlib
micromamba_root = '/content/micromamba'
os.environ['MAMBA_ROOT_PREFIX'] = micromamba_root
if not Path('/content/bin/micromamba').exists():
    subprocess.run("curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content bin/micromamba", shell=True, check=True)
MM = '/content/bin/micromamba'
subprocess.run([MM, 'create', '-y', '-n', 'pepita', 'python=3.10', 'pip'], check=True)
subprocess.run([MM, 'run', '-n', 'pepita', 'python', '-m', 'pip', 'install', '--upgrade', 'pip==24.0'], check=True)
packages = [
    'torch==1.13.1+cu117', 'torchvision==0.14.1+cu117',
    'pytorch-lightning==1.6.5', 'torchmetrics==0.9.2',
    'numpy==1.21.2', 'Pillow==8.4.0', 'scikit-learn==1.2.1', 'scipy==1.10.1',
    'loguru==0.6.0', 'setuptools==80.9.0', 'ray[tune]==2.3.1', 'yacs==0.1.8',
    'tensorboard==2.12.3', 'protobuf<4'
]
subprocess.run([MM, 'run', '-n', 'pepita', 'python', '-m', 'pip', 'install', '--extra-index-url', 'https://download.pytorch.org/whl/cu117', *packages], check=True)
print('Environment setup complete.')

## Check GPU support and fix the zero-validation-loader edge case

The project builds a validation loader even when `val_split=0`; in that case the split is `None`. This guarded change leaves the full training dataset untouched and avoids constructing a loader from `None`. The trainer’s validation method already uses the separate test loader.

In [ ]:
from pathlib import Path
import subprocess, textwrap
patch_script = r'''from pathlib import Path
p = Path('pepita/dataset/getdataloader.py')
s = p.read_text(encoding='utf-8')
old = '    val_dataloader = DataLoader(val_data, batch_size=batchsize, shuffle=False, num_workers=num_workers, pin_memory=True)\n'
new = '    val_dataloader = DataLoader(val_data, batch_size=batchsize, shuffle=False, num_workers=num_workers, pin_memory=True) if val_data is not None else None\n'
count = s.count(old)
if count != 3:
    raise RuntimeError(f'Expected three matching validation-loader lines; found {count}. Source differs from the reviewed project version; stopping rather than applying an uncertain patch.')
p.write_text(s.replace(old, new), encoding='utf-8')
print('Applied guarded val_split=0 loader fix in', p)
'''
subprocess.run([MM, 'run', '-n', 'pepita', 'python', '-c', patch_script], cwd=PROJECT, check=True)

check_script = """import torch, pytorch_lightning as pl
cap = torch.cuda.get_device_capability(0)
arch = torch.cuda.get_arch_list()
print('PyTorch', torch.__version__, 'Lightning', pl.__version__)
print('CUDA available', torch.cuda.is_available())
print('GPU', torch.cuda.get_device_name(0), 'capability', cap)
print('PyTorch GPU architectures', arch)
required = f'sm_{cap[0]}{cap[1]}'
if required not in arch:
    raise SystemExit(f'Unsupported GPU for this PyTorch build: {required} is not in {arch}. Select T4 or A100; do not start training on this GPU.')
x = torch.randn(512, 512, device='cuda')
print('GPU operation check', float((x @ x).mean()))
"""
result = subprocess.run([MM, 'run', '-n', 'pepita', 'python', '-c', check_script], cwd=PROJECT, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('The environment check failed. Read the import or CUDA error above and share it before starting training.')

## Run CIFAR-100

The run uses seed `20261031`. CIFAR-100 will download into the project’s `datasets/` folder. Keep this notebook connected until training finishes; the project logs the epoch schedule and writes a YAML configuration and TensorBoard event file under `experiments/cifar100_pepita_1hidden/`.

In [ ]:
from datetime import datetime, timezone
import subprocess, sys
experiment = 'cifar100_pepita_1hidden'
seed = 20261031
cmd = [MM, 'run', '-n', 'pepita', 'python', 'scripts/train.py',
       '-en', experiment, '-d', 'cifar100', '-ls', '1024',
       '-lr', '0.01', '-wd', '0.00001', '-e', '100', '-bs', '64',
       '-mom', '0.9', '-dr', '0.1', '-lrd', '0.1', '-de', '60', '90',
       '-li', 'he_uniform', '-bi', 'uniform', '-bstd', '0.05',
       '-w', '2', '-s', str(seed)]
run_started_utc = datetime.now(timezone.utc).isoformat()
print('Started UTC:', run_started_utc)
print('Command:', ' '.join(cmd))
log_path = PROJECT / 'experiments' / experiment / 'colab_console.log'
log_path.parent.mkdir(parents=True, exist_ok=True)
with open(log_path, 'w', encoding='utf-8') as log:
    proc = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
        log.write(line)
    code = proc.wait()
run_finished_utc = datetime.now(timezone.utc).isoformat()
print('Finished UTC:', run_finished_utc)
if code != 0:
    raise RuntimeError(f'Training exited with status {code}. The console log is at {log_path}')
print('Training finished successfully.')

## Save the run record

This packages the saved experiment configuration, training log, TensorBoard events, and a runtime record. Download the ZIP and keep it with the capstone results.

In [ ]:
from datetime import datetime, timezone
import subprocess, json, shutil
info_cmd = "import torch, pytorch_lightning as pl, platform, json; print(json.dumps({'python':platform.python_version(),'pytorch':torch.__version__,'torchvision':__import__('torchvision').__version__,'pytorch_lightning':pl.__version__,'cuda_runtime':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'gpu_capability':torch.cuda.get_device_capability(0)}))"
runtime = subprocess.check_output([MM, 'run', '-n', 'pepita', 'python', '-c', info_cmd], cwd=PROJECT, text=True).strip()
(run_dir / 'runtime_info.json').write_text(json.dumps({'runtime':json.loads(runtime),'seed':seed,'started_utc':run_started_utc,'finished_utc':run_finished_utc,'saved_utc':datetime.now(timezone.utc).isoformat()}, indent=2), encoding='utf-8')
(run_dir / 'runtime_info.json').write_text(json.dumps({'runtime':json.loads(runtime),'seed':seed,'saved_utc':datetime.now(timezone.utc).isoformat()}, indent=2), encoding='utf-8')
archive_base = Path('/content') / experiment
archive = shutil.make_archive(str(archive_base), 'zip', root_dir=PROJECT, base_dir=str(Path('experiments') / experiment))
print('Run archive:', archive)
files.download(archive)